In [ ]:
import pde_descriptions,system_prompt
from parse_agent.prompt import parse_prompt
from code_agent.prompt import code_prompt
from debug_agent.prompt import debug_prompt

from dotenv import load_dotenv
import json
from ollama import chat
from verify_script.verify_result import *

advection_description = pde_descriptions.advection_description
twoD_reaction_diffusion_description = pde_descriptions.twoD_reaction_diffusion_description
pde_description = twoD_reaction_diffusion_description.format(nu = 0.1, p = 0.1)
#system_prompt = system_prompt.system_prompt
user_prompt = parse_prompt.format(
    user_input=pde_description,
)

response = chat(
    model='qwen3:8b',
    messages=[
        #{'role': 'system', 'content': system_prompt},po
        {'role': 'user', 'content': user_prompt},
    ],
)
print(response.message.content)

#deleat ```json at beginning of response.text if exist
response_text = response.message.content
if response_text.startswith("```json"):
    response_text = response_text[len("```json"):]

# #deleat ``` at end of response.text if exist
if response_text.endswith("```"):
    response_text = response_text[:-3]

with open("./result/parsed_resp.json", "w", encoding="utf-8") as f:
    f.write(response_text)

# read json
with open("./result/parsed_resp.json", "r") as f:
    parsed_resp = json.load(f)
print(parsed_resp.keys())

In [2]:
number_of_state_variables = parsed_resp["number_of_state_variables"]
texture_size = parsed_resp["texture_size"]
spatial_step = parsed_resp["spatial_step"]
temporal_step = parsed_resp["temporal_step"]
time_horizon = parsed_resp["time_horizon"]
#initial_condition_file = '"C:\\Users\\xan37\\OneDrive - Georgia Institute of Technology\\Documents\\GitHub\\cardiac-agent\\baselines models\\fk_data\\tau_d_0.5714\\IC.csv"'
parameter_values = parsed_resp["parameter_values"]
parameter_values
parameter_str = "\n".join([
    f"    const float {k:} = {v};" 
    for k, v in parameter_values.items()
])

coding_skeleton_file = f"./skeleton_script/{parsed_resp['number_of_state_variables']}V/skeleton.html"
# Load the file
with open(coding_skeleton_file, 'r') as f:
    html_template = f.read()

# Replace placeholders with your Python variables
updated_html = html_template.replace('{{DT_VALUE}}', str(temporal_step)).replace('{{DX_VALUE}}', str(spatial_step))
updated_html = updated_html.replace('{{TEXTURE_VALUE}}', str(texture_size))
# Optional: Save it back to a new file
with open('./result/skeleton.html', 'w') as f:
    f.write(updated_html)
    

In [3]:
with open(f"./skeleton_script/{parsed_resp['number_of_state_variables']}V/march_skeleton.frag", 'r') as f:
    coding_skeleton = f.read()
updated_coding_skeleton = coding_skeleton.replace('{{PARAMETER_VALUES}}', parameter_str)
user_prompt = code_prompt.format(
    PDEs=parsed_resp["PDEs"],
    coding_skeleton = updated_coding_skeleton
)

In [ ]:
response = chat(
    #model = 'gemma4:31b',
    model='qwen3:8b',
    messages=[
        #{'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ],
)
print(response.message.content)


In [5]:
#deleat ```glsl at beginning of response.text if exist
response_text = response.message.content
if response_text.startswith("```glsl"):
    response_text = response_text[len("```glsl"):]

# #deleat ``` at end of response.text if exist
if response_text.endswith("```"):
    response_text = response_text[:-3]  

with open("./result/march_shader.frag", "w", encoding="utf-8") as f:
    f.write(response_text)


In [6]:
# replace march shader script in skeleton.html with the generated march shader code
with open('./result/skeleton.html', 'r') as f:
    html_content = f.read()
with open('./result/march_shader.frag', 'r') as f:    march_shader_code = f.read()
updated_html = html_content.replace('{{MARCH_SHADER_CODE}}', march_shader_code)
# save to new html
with open('./result/updated_skeleton.html', 'w') as f:
    f.write(updated_html)

## verify result

In [27]:
%load_ext autoreload
%autoreload 2
from verify_script.verify_result import *

IC_file = "./random_sample_t0.csv"
simulation_file = './result/updated_skeleton.html'
logs = verify_result(simulation_file,IC_file,T_end=2)

if logs != "Success":
    with open("browser_logs.txt", "w", encoding="utf-8") as f:
        for entry in logs:
            # Format: [LEVEL] Timestamp - Message
            line = f"[{entry['level']}] {entry['timestamp']} - {entry['message']}\n"
            f.write(line)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Serving ./result/updated_skeleton.html at http://localhost:8001/updated_skeleton.html
Simulation started on localhost. Monitoring console...
Could not find or click the button automatically: Message: 

{'level': 'SEVERE', 'message': 'https://abubujs.org/libs/Abubu.latest.js 14739:24 "No successful compilation is available"', 'source': 'console-api', 'timestamp': 1776709521041}
Shutting down...


In [4]:
%load_ext autoreload
%autoreload 2
from debug_agent.prompt import debug_prompt


with open('browser_logs.txt', 'r', encoding='utf-8') as f:
    original_log_content = f.read()

with open("./result/march_shader.frag", "r", encoding="utf-8") as f:
    shader_codes = f.read()
debug_prompt = debug_prompt.format(
    shader_codes = shader_codes,
    log_info = original_log_content
)

debug_response = chat(
    model='qwen3:8b',
    messages=[
        #{'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': debug_prompt},
    ],
)
print(debug_response.message.content)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
```glsl
#version 300 es
precision highp float;
// precision highp int;

uniform sampler2D inTexture;
uniform float dt, dx, dy;
uniform float D_u, D_v, k;

in vec2 cc, pixPos;

layout (location = 0) out vec4 ocolor;

#define u       color.r
#define v       color.g

void main() {
    vec2 size = vec2(textureSize(inTexture, 0));
    vec2 ii = vec2(1.,0.)/size;
    vec2 jj = vec2(0.,1.)/size;

    vec4 color = texture(inTexture, cc);
    float u_old = color.r;
    float v_old = color.g;

    float laplacian_u = 0.0;
    float u_right = texture(inTexture, cc + ii).r;
    float u_left = texture(inTexture, cc - ii).r;
    laplacian_u += (u_right + u_left - 2.0 * u_old) / (dx*dx);

    float u_top = texture(inTexture, cc - jj).r;
    float u_bottom = texture(inTexture, cc + jj).r;
    laplacian_u += (u_top + u_bottom - 2.0 * u_old) / (dy*dy);

    float laplacian_v = 0.0;
    float v_right = texture(inTextu

In [ ]:
#deleat ```glsl at beginning of response.text if exist
response_text = debug_response.message.content
if response_text.startswith("```glsl"):
    response_text = response_text[len("```glsl"):]
# deleat #version 300 es at beginning of response.text if exist
if response_text.startswith("#version 300 es"):
    response_text = response_text[len("#version 300 es"):].lstrip()  # also remove leading whitespace

# #deleat ``` at end of response.text if exist
if response_text.endswith("```"):
    response_text = response_text[:-3]

with open("march_shader.frag", "w", encoding="utf-8") as f:
    f.write(response_text)